# Training StarDist for Lipid Droplet Segmentation in SRS Images

This notebook trains a StarDist 2D instance-segmentation model for
automated detection and quantitative analysis of lipid droplets in
stimulated Raman scattering (SRS) microscopy images.

## Workflow

SRS images  
→ Manual instance annotation  
→ Image normalization  
→ Training/validation split  
→ StarDist training  
→ Threshold optimization  
→ Validation  
→ Testing on unseen SRS images  
→ Lipid-droplet quantification

Manual instance annotations were generated using Labkit in Fiji/ImageJ.

Each lipid droplet is represented by an independent integer label:

- 0 = background
- 1 = lipid droplet 1
- 2 = lipid droplet 2
- 3 = lipid droplet 3
- ...

The trained model is saved as `lipid_droplet_v1`.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tifffile import imread
from csbdeep.utils import normalize
from stardist.models import Config2D, StarDist2D
from skimage.measure import regionprops_table

print("Packages loaded successfully.")

Packages loaded successfully.


In [ ]:
# ============================================================
# PROJECT DIRECTORIES
# ============================================================

# This notebook is expected to be inside:
# SRS_LipidDroplet_StarDist/notebooks/

PROJECT_DIR = Path("..")

IMAGE_DIR = PROJECT_DIR / "data" / "train_images"
MASK_DIR = PROJECT_DIR / "data" / "train_masks"
TEST_DIR = PROJECT_DIR / "examples"
MODEL_DIR = PROJECT_DIR / "model"

MODEL_NAME = "lipid_droplet_v1"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory :", PROJECT_DIR.resolve())
print("Training images   :", IMAGE_DIR.resolve())
print("Training masks    :", MASK_DIR.resolve())
print("Test images       :", TEST_DIR.resolve())
print("Model output      :", MODEL_DIR.resolve())

In [ ]:
# ============================================================
# LOAD TRAINING IMAGES AND ANNOTATIONS
# ============================================================

image_files = sorted(IMAGE_DIR.glob("*.tif"))
mask_files = sorted(MASK_DIR.glob("*.tif"))

print("Images found:", len(image_files))
print("Masks found :", len(mask_files))


print("\nTraining images:")

for f in image_files:
    print(" -", f.name)


print("\nAnnotation masks:")

for f in mask_files:
    print(" -", f.name)


# Basic checks

if len(image_files) == 0:
    raise FileNotFoundError(
        f"No training images were found in:\n{IMAGE_DIR.resolve()}"
    )

if len(image_files) != len(mask_files):
    raise ValueError(
        "The number of training images and masks does not match."
    )


# Load images

X = [
    imread(str(f))
    for f in image_files
]

Y = [
    imread(str(f))
    for f in mask_files
]


print("\nLoaded", len(X), "images")
print("Loaded", len(Y), "masks")

In [ ]:
# ============================================================
# CHECK IMAGE / MASK PAIRS
# ============================================================

for i, (img, mask) in enumerate(zip(X, Y)):

    labels = np.unique(mask)
    droplet_labels = labels[labels > 0]

    print(f"\nPair {i + 1}")
    print("Image:", image_files[i].name)
    print("Mask :", mask_files[i].name)

    print("Image shape:", img.shape)
    print("Mask shape :", mask.shape)

    print("Image dtype:", img.dtype)
    print("Mask dtype :", mask.dtype)

    print(
        "Annotated droplets:",
        len(droplet_labels)
    )

    assert img.shape == mask.shape, (
        f"Image and mask shapes do not match for pair {i + 1}."
    )

    assert mask.min() >= 0, (
        f"Mask {i + 1} contains negative labels."
    )

print("\nAll image-mask pairs passed the basic checks.")

In [ ]:
# ============================================================
# VISUALIZE TRAINING ANNOTATIONS
# ============================================================

for i in range(len(X)):

    img = X[i]
    mask = Y[i]

    droplet_count = len(
        np.unique(mask)[np.unique(mask) > 0]
    )

    mask_overlay = np.ma.masked_where(
        mask == 0,
        mask
    )

    plt.figure(figsize=(15, 5))


    # Original SRS image

    plt.subplot(1, 3, 1)

    plt.imshow(
        img,
        cmap="gray"
    )

    plt.title(
        f"Original Image {i + 1}"
    )

    plt.axis("off")


    # Manual annotation

    plt.subplot(1, 3, 2)

    plt.imshow(
        mask,
        cmap="nipy_spectral"
    )

    plt.title(
        f"Manual Annotation\n"
        f"{droplet_count} droplets"
    )

    plt.axis("off")


    # Overlay

    plt.subplot(1, 3, 3)

    plt.imshow(
        img,
        cmap="gray"
    )

    plt.imshow(
        mask_overlay,
        cmap="nipy_spectral",
        alpha=0.5
    )

    plt.title(
        "SRS + Annotation"
    )

    plt.axis("off")


    plt.tight_layout()
    plt.show()

## Image Normalization

SRS images are normalized using the 1st and 99.8th intensity
percentiles.

The same normalization procedure should be applied to new images
during inference.

In [ ]:
# ============================================================
# NORMALIZE IMAGES
# ============================================================

X_norm = [
    normalize(
        img,
        1,
        99.8
    )
    for img in X
]

print(
    f"Normalized {len(X_norm)} images."
)

In [ ]:
# ============================================================
# TRAINING / VALIDATION SPLIT
# ============================================================

# Image 0 is reserved for validation.
# Images 1-15 are used for training.
#
# This corresponds to:
# 15 training images
# 1 validation image

X_train = X_norm[1:16]
Y_train = Y[1:16]

X_val = X_norm[:1]
Y_val = Y[:1]


print(
    "Training images:",
    len(X_train)
)

print(
    "Validation images:",
    len(X_val)
)

### Dataset split

For the current `lipid_droplet_v1` model:

- **15 images** are used for training.
- **1 image** is reserved for validation.

Because the current validation set contains only one image, validation
performance should be interpreted cautiously. Future model versions
should use a larger independent validation/test dataset.

In [ ]:
# ============================================================
# STARDIST CONFIGURATION
# ============================================================

config = Config2D(

    # Number of radial directions used to represent each object
    n_rays=32,

    # Preserve full image resolution
    grid=(1, 1),

    # CPU training
    use_gpu=False,

    # Training patch dimensions
    train_patch_size=(256, 256),

    # Number of patches per batch
    train_batch_size=4,

    # Learning rate
    train_learning_rate=0.0003
)

print(config)

In [ ]:
# ============================================================
# CREATE MODEL
# ============================================================

model = StarDist2D(
    config,
    name=MODEL_NAME,
    basedir=str(MODEL_DIR)
)

print("Model name:", MODEL_NAME)

print(
    "Model directory:",
    (MODEL_DIR / MODEL_NAME).resolve()
)

In [ ]:
# ============================================================
# TRAIN MODEL
# ============================================================

history = model.train(
    X_train,
    Y_train,
    validation_data=(
        X_val,
        Y_val
    ),
    epochs=100,
    steps_per_epoch=50
)

print("\nTraining complete.")

In [ ]:
# ============================================================
# PLOT TRAINING HISTORY
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "StarDist Training History"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# OPTIMIZE DETECTION THRESHOLDS
# ============================================================

model.optimize_thresholds(
    X_val,
    Y_val
)

print(
    "Detection thresholds optimized."
)

In [ ]:
# ============================================================
# PREDICT VALIDATION IMAGE
# ============================================================

test_img = X_val[0]
true_mask = Y_val[0]


pred_mask, details = model.predict_instances(
    test_img
)


manual_count = len(
    np.unique(true_mask)[
        np.unique(true_mask) > 0
    ]
)


predicted_count = len(
    np.unique(pred_mask)[
        np.unique(pred_mask) > 0
    ]
)


print(
    "Manual droplets:",
    manual_count
)

print(
    "Predicted droplets:",
    predicted_count
)

In [ ]:
# ============================================================
# VISUAL VALIDATION
# ============================================================

manual_overlay = np.ma.masked_where(
    true_mask == 0,
    true_mask
)

prediction_overlay = np.ma.masked_where(
    pred_mask == 0,
    pred_mask
)


plt.figure(
    figsize=(15, 5)
)


# Original image

plt.subplot(
    1,
    3,
    1
)

plt.imshow(
    test_img,
    cmap="gray"
)

plt.title(
    "SRS Image"
)

plt.axis("off")


# Manual annotation

plt.subplot(
    1,
    3,
    2
)

plt.imshow(
    test_img,
    cmap="gray"
)

plt.imshow(
    manual_overlay,
    cmap="nipy_spectral",
    alpha=0.5
)

plt.title(
    f"Manual Annotation\n"
    f"{manual_count} droplets"
)

plt.axis("off")


# StarDist prediction

plt.subplot(
    1,
    3,
    3
)

plt.imshow(
    test_img,
    cmap="gray"
)

plt.imshow(
    prediction_overlay,
    cmap="nipy_spectral",
    alpha=0.5
)

plt.title(
    f"StarDist Prediction\n"
    f"{predicted_count} droplets"
)

plt.axis("off")


plt.tight_layout()
plt.show()